# TNT hybrid M1 pretraining gate

This experiment answers one question before any further ablation: **does the full hybrid improve on TNT V1 when both are pretrained on M1 and fine-tuned on Gold-train under the same budget?**

Protocol: 3,330 matched M1 pairs, 20 synthetic epochs, 20 Gold epochs, seed 2026, checkpoint selection on Gold-dev every five epochs. Gold-test is disabled. The V2 operation/boundary/word class weights are calculated once from Gold-train and frozen before M1 training. The bounded segment head uses 16 slots: the observed maximum is 14 across M1 plus Gold-train, while 99% of pairs require at most two.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    candidates = [parent for parent in PROJECT_ROOT.parents if (parent / 'src').is_dir()]
    if not candidates:
        raise FileNotFoundError('Run from the PFE repository or one of its subdirectories.')
    PROJECT_ROOT = candidates[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from IPython.display import display

from src.modeling.tnt_hybrid_m1_gate import (
    BENCHMARK_NAME,
    RESULTS_DIR,
    gate_decision,
    gate_prediction_comparison,
    gate_protocol,
    gate_summary,
    prepare_gate_data,
    prepare_gate_initialization,
    run_m1_gate,
)
from src.modeling.tnt_hybrid_diagnostics import (
    DIAGNOSTIC_DIR,
    run_head_diagnostics,
)

print('Project:', PROJECT_ROOT)
print('Experiment:', BENCHMARK_NAME)
print('Results:', RESULTS_DIR.resolve())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Project: C:\Users\thioy\Desktop\School\PFE
Experiment: tnt_hybrid_m1_gate_v2
Results: C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_hybrid_m1_gate_v2
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 1. Recreate the controlled data selection

This uses the same common-target sampling policy as the frozen benchmark. All retained methods are loaded only to recreate the shared 3,330 formal-target pool and character vocabulary; **only M1 is trained**.

In [3]:
PROTOCOL = gate_protocol()
gold, selected, dataset_audit = prepare_gate_data(PROTOCOL)
display(pd.DataFrame([PROTOCOL.__dict__]))
display(dataset_audit)
print('M1 selected rows:', len(selected['m1']))
print('Gold train/dev rows:', len(gold['train']), len(gold['dev']))

,seed,matched_unique_rows,synthetic_epochs,gold_epochs,synthetic_learning_rate,gold_learning_rate,synthetic_batch_size,gold_batch_size,evaluation_batch_size,generated_eval_interval_epochs
0,2026,3330,20.0,20.0,0.0003,0.0001,64,32,64,5


,method,loaded_rows,unique_pair_rows,unique_target_rows,selected_rows,common_target_pool,seed,dev_input_overlap,dev_pair_overlap,test_input_overlap,test_pair_overlap,source_over_255_chars,target_over_255_chars
0,m1,17777,16377,16375,3330,3430,2026,0,0,0,0,8,6
1,m2,17777,16544,16375,3330,3430,2026,0,0,0,0,10,6
2,m3,17777,16384,16375,3330,3430,2026,0,0,0,0,8,6
3,m4,3433,3433,3430,3330,3430,2026,0,0,0,0,7,6
4,m6,17777,16427,16375,3330,3430,2026,0,0,0,0,6,6


M1 selected rows: 3330
Gold train/dev rows: 142 30


## 2. Prepare the full-hybrid initialization

This creates the model and tokenizer but does not train them. Re-running the cell reuses the fingerprinted initialization.

In [4]:
shared_initialization = prepare_gate_initialization(gold, selected, PROTOCOL)
print('Initialization:', shared_initialization.resolve())

[TNT-V2] Reusing hybrid initialization: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_m1_gate_v2\shared_initialization\tnt_hybrid_char_e702b9f11b13
Initialization: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_m1_gate_v2\shared_initialization\tnt_hybrid_char_e702b9f11b13


## 3. Run M1 synthetic pretraining → Gold fine-tuning

The cell prints tqdm progress, evaluation metrics and both stages. Interrupted checkpoints are resumed when possible. Leave `RERUN=False` to reuse a completed gate; use `True` only when intentionally creating a fresh attempt.

In [5]:
RUN_M1_GATE = True
RERUN = False

if RUN_M1_GATE:
    m1_gate_selection = run_m1_gate(
        gold, selected, shared_initialization, PROTOCOL, rerun=RERUN
    )
    display(m1_gate_selection.get('development_metrics', {}))
else:
    print('Skipped. Set RUN_M1_GATE=True when ready to train.')


[M7] START m1 / synthetic_pretrain / seed 2026 / attempt 002 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_m1_gate_v2\m1\seed_2026\synthetic_pretrain\attempt_002
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_m1_gate_v2\shared_initialization\tnt_hybrid_char_e702b9f11b13


You are using a model of type `wolof_tnt_hybrid_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/46 [00:00<?, ?it/s]

[M7] Loaded 1,779,274 total parameters; training 1,779,274 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


Tokenizing TNT edit pairs:   0%|          | 0/3330 [00:00<?, ? examples/s]

...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_m1_gate_v2\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_m1_gate_v2\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_m1_gate_v2\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridTransformerForConditionalGeneration.forward` and have been ignored: source_id, target_text, source_text. If source_id, target_text, source_text are not expected by `TntHybridTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m1 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.20044709388971685
[compact benchmark] m1: continuous 20-epoch gold run complete; selected dev CER=0.20044709388971685


{'rows': 30,
 'cer': 0.20044709388971685,
 'wer': 0.6610169491525424,
 'chrf': 56.46345131346913,
 'exact_match': 0.0,
 'correction_tp': 2,
 'correction_fp': 76,
 'correction_fn': 70,
 'correction_precision': 0.02564102564102564,
 'correction_recall': 0.027777777777777776,
 'correction_f1': 0.026666666666666665,
 'overcorrection_rate': 0.9743589743589743,
 'unchanged_reference_rows': 0,
 'unchanged_sentence_overcorrection_rate': 0.0}

## 4. Gate decision on raw Gold-dev references

The table recomputes the frozen TNT V1 M1 metrics against the original Gold-dev text, avoiding the former decoded-reference issue. The strict gate requires non-worse CER, improved WER and correction F1, reduced overcorrection, and at least one true correction.

In [6]:
summary = gate_summary(gold['dev'])
preferred = [
    'condition', 'cer', 'wer', 'chrf', 'exact_match',
    'correction_tp', 'correction_fp', 'correction_fn',
    'correction_precision', 'correction_recall', 'correction_f1',
    'overcorrection_rate',
]
display(summary[[column for column in preferred if column in summary.columns]])
display(gate_decision(summary))

,condition,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate
0,identity,0.233234,0.630508,56.052143,0.0,0,0,72,0.000000,0.000000,0.000000,0.000000
1,tnt_v1_m1,0.181073,0.667797,55.888125,0.0,2,67,70,0.028986,0.027778,0.028369,0.971014
2,tnt_hybrid_m1,0.200447,0.661017,56.463451,0.0,2,76,70,0.025641,0.027778,0.026667,0.974359


{'ready': True,
 'proceed_to_ablation': False,
 'criteria': {'cer_not_worse_than_v1': False,
  'wer_improved_over_v1': True,
  'correction_f1_improved': False,
  'overcorrection_reduced': False,
  'at_least_one_true_correction': True},
 'note': 'A failed strict gate triggers head diagnostics, not Gold-test evaluation or automatic full-benchmark training.'}

## 5. Qualitative comparison

Inspect whether metric changes correspond to correct normalization rather than more edits or identity copying.

In [7]:
comparison = gate_prediction_comparison(gold['dev'])
display(comparison)

,source_id,source_text,reference,prediction_tnt_v1_m1,prediction_tnt_hybrid_m1
0,gold:dev:10000,​ té dou wakhé dank,té du waxee ndànk,te du waxe dank,te du waxe daank
1,gold:dev:10003,cachet wam mo wara diekh. sokhna ak salimata l...,cachet wam moo wara jeex sokhna ak salimata lé...,cachet wam mo wara djeex. soxna ak salimata le...,cachet wam moo waara jeex. soxna ak salimata l...
2,gold:dev:10005,nimako diépé sakh da eupeu,ni ma ko jéppe sax da ëpp,nimako diépé sax da ëpë,nimako diépe sax da ëpë
3,gold:dev:10007,maman oumy mome rk lay bagna guis,maman oumy moom rek laay baña gis,maman oumy mome rk lay bañña gis,maman oumy moom rk lay baaa ggis
4,gold:dev:10008,mére bi mo sof ba paré reuy lamign,mère bi moo soof ba pare réy làmmiñ,mére bi mo sof ba pare rëy lamigñ,mére bi moo soof ba pare rëy lamie
5,gold:dev:10013,franchement khady thiam nak dafa eupeul fi kou...,franchement khady thiam nak dafa ëppal fi kuy ...,franchement hady ciam nak dafa ëpël fi kuy sëy...,franceement xady cam nak dafa ëpël fi kuy sëy ...
6,gold:dev:10014,loutakh pere ndiouga di raamal vieux?alors que...,lu tax pere ndiouga di raamal vieux alors que ...,lutax per ndjouga di raamal viëëx?alors qu ñoo...,utax pere njñuga i raamal vjëx?aloors q ñoo bo...
7,gold:dev:10020,ki di way baabel voix bi nèx na deh kanla,ki di woy baabel voix bi neex na de kan la,ki di way baabel voix bi nèx na dnx kanla,ki di waay baabel voix bi nèx na dne kanla
8,gold:dev:10021,maman omou kiye rayé si série douko ray,maman omou kuy reye ci série du ko rey,maman omou kiy raye si sérin duko ray,maman omu kiyé raye si serjc duko ray
9,gold:dev:10026,le meilleurs séries kou d'accord lalal j'aime ️,les meilleurs séries ku d'accord laalal j'aime,le meilleërs séries ku d'accord lalal j'aim ️,le meillërs seeries ku d'accord lalal j'aimé ️


## 6. Diagnose the trained heads

This is read-only inference on Gold-dev. It exports raw and effective character-operation predictions, boundary decisions, word KEEP/EDIT probabilities, confusion matrices, and lookup-based French/entity/protection subsets. It does not retrain or evaluate Gold-test.

In [ ]:
head_summary = run_head_diagnostics(gold['dev'])
print('Saved under:', DIAGNOSTIC_DIR.resolve())
display(pd.DataFrame(head_summary['operation']['raw_per_class']))
display(pd.DataFrame(head_summary['operation']['effective_per_class']))
display(pd.DataFrame(head_summary['boundary']['per_class']))
display(pd.DataFrame(head_summary['word']['per_class']))
display({
    'operation_confidence': head_summary['operation']['confidence'],
    'segment': head_summary['segment'],
    'boundary_actions': head_summary['boundary']['accepted_actions'],
    'word_keep_gate_rate': head_summary['word']['keep_gate_rate'],
    'gold_edit_incorrectly_blocked': head_summary['word']['gold_edit_incorrectly_blocked'],
    'known_french': head_summary['word']['known_french'],
    'known_entities': head_summary['word']['known_entities'],
    'known_protected': head_summary['word']['known_protected'],
})

In [ ]:
word_diagnostics = pd.read_csv(DIAGNOSTIC_DIR / 'word_predictions.csv', keep_default_na=False)
print('Gold KEEP words not protected by the confidence gate:')
display(
    word_diagnostics.loc[
        word_diagnostics['gold_word_action'].eq('keep')
        & ~word_diagnostics['keep_gate_applied']
        [
            'source_id', 'source_word', 'keep_probability',
            'raw_character_edit_positions', 'lookup_language',
            'lookup_entity_type', 'lookup_protected',
        ],
    ].sort_values(['raw_character_edit_positions', 'keep_probability'], ascending=[False, True]).head(30)
)

print('Gold EDIT words incorrectly blocked by the KEEP gate:')
display(
    word_diagnostics.loc[
        word_diagnostics['gold_word_action'].eq('edit')
        & word_diagnostics['keep_gate_applied']
    ].sort_values('keep_probability', ascending=False).head(30)
)